# 01 · Why Orchestration: Where the Agent Loop Breaks

> **Kernel:** `Phase 5 · Concepts` · **Plain Python:** no framework and no
> earlier phase; everything is defined in this notebook. A scripted model makes
> every failure happen on purpose. One optional cell at the end uses Ollama.

An **agent** is a loop: ask the model what to do, run the tool it picks, feed the
result back, repeat. For many problems that loop is exactly right.

This notebook covers the problems where it **isn't**. You'll write the loop,
then break it in five specific, reproducible ways. Each break points at one idea
that every orchestration framework in this phase is built around, so you'll know
*why* orchestration exists before you learn how any framework does it.

**You will learn**
- The two things a plain agent loop hides: **who decides control flow** and **where state lives**
- Five failure modes: *order*, *parallelism*, *durability*, *waiting for humans*, *inspect & replay*
- The four building blocks that fix them: **state, nodes, edges, checkpoints**
- When orchestration is overkill, and the price you pay for it

**Prerequisites:** Python, and the idea of an LLM calling a tool (function calling).

## 1 · The agent loop (the thing we're going to break)

Below is a complete tool-using agent in about 40 lines. Read it carefully, since
the rest of the notebook is about its weaknesses.

- A **model** is any function that takes the conversation and returns a `Turn`:
  either final text, or a list of tool calls it wants made.
- A **tool** is a Python function plus a JSON-schema description the model reads.
- `agent_loop` runs *model → tools → model → …* until the model answers.

In [ ]:
from dataclasses import dataclass, field
import time, json, inspect, threading, pathlib
from concurrent.futures import ThreadPoolExecutor


@dataclass
class Turn:
    '''What a model returns: final text, OR tool calls it wants us to make.'''
    text: str = ''
    tool_calls: list = field(default_factory=list)   # [{'name': ..., 'args': {...}}]


@dataclass
class Tool:
    fn: callable
    schema: dict                     # what the model sees (name, description, parameters)


def tool(registry, description, **params):
    '''Decorator: register fn in `registry` with a JSON schema built from `params`.'''
    def wrap(fn):
        registry[fn.__name__] = Tool(fn, {
            'type': 'function',
            'function': {'name': fn.__name__, 'description': description,
                         'parameters': {'type': 'object',
                                        'properties': {k: {'type': v} for k, v in params.items()},
                                        'required': list(params)}}})
        return fn
    return wrap


def agent_loop(model, tools, task, system='You are a helpful agent.', approve=None, max_steps=8):
    messages = [{'role': 'system', 'content': system},      # ← ALL the state lives here,
                {'role': 'user', 'content': task}]          #   in one local variable
    trace = []
    for _ in range(max_steps):
        turn = model(messages, [t.schema for t in tools.values()])   # ← the MODEL decides what's next
        if not turn.tool_calls:
            trace.append(('ANSWER', turn.text))
            return turn.text, trace
        messages.append({'role': 'assistant', 'content': turn.text,
                         'tool_calls': [{'function': {'name': c['name'], 'arguments': c['args']}}
                                        for c in turn.tool_calls]})
        for call in turn.tool_calls:                        # ← one call at a time
            name, args = call['name'], call['args']
            trace.append(('ACTION', f'{name}({args})'))
            if approve is not None and not approve(name, args):
                result = f'Denied: a human rejected {name}.'
            else:
                try:
                    result = str(tools[name].fn(**args))
                except Exception as err:                    # a tool error becomes an observation
                    result = f'Error: {err}'
            trace.append(('OBSERVATION', result))
            messages.append({'role': 'tool', 'content': result, 'tool_name': name})
    return 'Stopped: step limit reached.', trace
    # returning (or crashing) → `messages` is gone


def show(trace):
    for kind, text in trace:
        print(f'   {kind:<12}{text}')

Read it with two questions in mind:

1. **Who decides what happens next?** The model, every single turn. The code
   never says "after X, always do Y". It only offers tools and runs what's asked.
2. **Where does the work live?** In `messages`, a Python list in one process's
   memory. When the function returns, or the process dies, the work is gone.

For a two-minute Q&A with a couple of tool calls, both are fine. Now imagine a
real job: *"Reproduce this bug, fix it, run the tests, wait for a reviewer to
approve, then merge."* That job needs guarantees:

| The job needs… | The plain loop gives… |
|---|---|
| Tests **always** run before merge | "The model will probably remember" |
| Three independent checks to run **at the same time** | One after another |
| Survive a crash at step 19 of 20 | Start again from step 1 (and pay again) |
| Wait **hours** for a reviewer | A blocked thread, or a denial |
| Rerun **just** the step that went wrong | Rerun everything |

**Orchestration** is the discipline of giving a multi-step AI system those
guarantees. Let's watch each gap happen.

## 2 · A scripted model, so failures happen on cue

A real model is non-deterministic: it *might* skip a step today and not
tomorrow. To show each failure reliably we use `ScriptedModel`, a fake that
replays a fixed list of turns. A turn is a string (final answer) or a list of
tool calls. It also counts its calls and tokens (600 in / 60 out per call, a
realistic agent turn), so we can measure wasted work.

`ProcessCrash` stands in for the process dying (`kill -9`, out-of-memory, a
redeploy). It subclasses `BaseException`, so the loop's `except Exception`
safety net can't catch it, just as no `try` can catch a real process death.

> Scripting the model isn't cheating: every failure below is one that real models
> produce in practice. The script just makes it happen every time.

In [ ]:
class ScriptedModel:
    TOKENS_PER_CALL = 600 + 60

    def __init__(self, turns):
        self.turns, self.calls = list(turns), 0

    def __call__(self, messages, tools):
        t = self.turns[self.calls] if self.calls < len(self.turns) else 'Done.'
        self.calls += 1
        return Turn(text=t) if isinstance(t, str) else Turn(tool_calls=t)

    @property
    def tokens(self):
        return self.calls * self.TOKENS_PER_CALL


class ProcessCrash(BaseException):
    '''The process died. Not a normal error: nothing in the loop can catch it.'''

## 3 · Five places the loop breaks

### 3.1 Order is a request, not a guarantee

**Rule:** never commit code unless the tests ran first. We put the rule in the
system prompt, which is the only place a plain loop *can* put it, and give the
agent both tools. The model (like real models on long tasks) goes straight to
`commit`.

In [ ]:
executed = []                        # what REALLY ran, in order
repo = {}

@tool(repo, 'Run the test suite and report pass/fail.')
def run_tests():
    executed.append('run_tests')
    return '12 passed, 0 failed'

@tool(repo, 'Commit the staged changes.', message='string')
def commit(message):
    executed.append('commit')
    return f'committed: {message!r}'

RULE = ('You are a release agent. RULE: you MUST call run_tests and see it pass '
        'BEFORE you call commit. Never commit untested code.')

model = ScriptedModel([[{'name': 'commit', 'args': {'message': 'fix login bug'}}],
                       'Committed the fix.'])
answer, trace = agent_loop(model, repo, 'Ship the login bug fix.', system=RULE)
show(trace)
print('\ntools actually executed, in order:', executed)

**What happened.** The rule lived in the prompt, and a prompt is *input to a
probabilistic model*, not a constraint on the program. Nothing in the loop
checks that `run_tests` happened; the loop runs whatever the model asks for.

**Could we patch it?** Yes: `approve` could veto `commit` unless `run_tests`
already ran. But then you're writing control flow ("B only after A") inside a
callback, scattered and invisible. Do that for ten rules and you have an
orchestrator, just a badly organised one.

**The idea it points to:** make order part of the **structure**. In a graph,
`commit` is reachable *only* through an edge from `run_tests`, so skipping it is
impossible rather than unlikely.

### 3.2 Everything runs one at a time

Three independent lookups, each taking 1 second (think: three slow APIs). The
model behaves well: it asks for all three **in one turn**, which invites the
program to run them together.

In [ ]:
weather = {}

@tool(weather, 'Current weather for a city (slow API, ~1s).', city='string')
def lookup(city):
    time.sleep(1.0)
    return f'{city}: 21°C, clear'

three = [{'name': 'lookup', 'args': {'city': c}} for c in ('Pune', 'Oslo', 'Lima')]

t0 = time.perf_counter()
agent_loop(ScriptedModel([three, 'All clear.']), weather, 'Weather in Pune, Oslo and Lima?')
print(f'agent loop : {time.perf_counter() - t0:.1f}s')

t0 = time.perf_counter()
with ThreadPoolExecutor() as pool:                       # the SAME function, run concurrently
    list(pool.map(lambda c: lookup(**c['args']), three))
print(f'in parallel: {time.perf_counter() - t0:.1f}s')

**What happened.** The lookups were independent, and Python can run them
concurrently, yet the loop took ~3s, because its *shape* is
`for call in turn.tool_calls: run(call)`. How work runs is decided by the
control structure, not by the tools.

It gets worse when the model asks in three separate turns: then you also pay for
three model round-trips, each re-reading the whole conversation.

**The idea it points to:** **fan-out / fan-in** as a first-class shape. One step
splits work into branches that run at the same time, and a later step joins
their results.

### 3.3 A crash throws everything away

A research agent fetches three sources (each a paid model turn), then writes a
report. While writing, the process dies: a deploy, an out-of-memory kill, a
laptop lid closing.

In [ ]:
fetched = []
crash_next_write = True
research = {}

@tool(research, 'Fetch and summarise one source.', topic='string')
def fetch(topic):
    fetched.append(topic)
    return f'notes on {topic}'

@tool(research, 'Write the final report from the gathered notes.')
def write_report():
    global crash_next_write
    if crash_next_write:
        crash_next_write = False
        raise ProcessCrash('process killed while writing the report')
    return 'report.md written'

SCRIPT = [[{'name': 'fetch', 'args': {'topic': 'checkpointing'}}],
          [{'name': 'fetch', 'args': {'topic': 'retries'}}],
          [{'name': 'fetch', 'args': {'topic': 'fan-out'}}],
          [{'name': 'write_report', 'args': {}}],
          'Report is ready.']

attempt1 = ScriptedModel(SCRIPT)
try:
    agent_loop(attempt1, research, 'Research orchestration and write a report.')
except ProcessCrash as err:
    print('💥', err)
print(f'attempt 1: {attempt1.calls} model calls, {attempt1.tokens} tokens, fetched {fetched}')

Where are the three fetched notes now? They were in `messages`, a local variable
inside `agent_loop`. The crash unwound that stack frame, so they're gone. The
only way forward is to start over:

In [ ]:
attempt2 = ScriptedModel(SCRIPT)
answer, report_trace = agent_loop(attempt2, research, 'Research orchestration and write a report.')
print('final answer:', answer)

total = attempt1.calls + attempt2.calls
print(f'model calls: {total} total, {attempt1.calls} of them repeated work ({attempt1.calls / total:.0%})')
print('fetch() ran for:', fetched)

**What happened.** Every completed step was redone: the same fetches, the same
model turns, the same tokens. With a cloud model that costs money; with slow
tools it costs minutes or hours; and with tools that have **side effects**
(sending an email, charging a card, opening a PR) redoing them is harmful.

*Retries* don't fix this. A retry re-runs one failed call inside a living
process. **Durability** means the progress itself outlives the process.

**The idea it points to:** a **checkpoint**. After every step, save the state
*and* the position ("steps 1–3 done, 4 is next") to durable storage. On
restart, load it and continue.

### 3.4 A human can't take their time

A common safety measure is an approval callback: before a risky tool runs, ask a
human. But look at *where* `approve` runs: inside the loop, in the middle of a
step. The loop can't continue until it returns, so the program has only two
choices: **block** until the human answers, or **give up**.

Our reviewer is at lunch. We give them 1.5 seconds (in real life, a timeout of
minutes) and then deny.

In [ ]:
reviewer_answered = threading.Event()        # nobody will set this: they're at lunch

def ask_reviewer(name, args):
    if name != 'commit':
        return True
    print(f'  ⏳ waiting for a reviewer to approve {name}({args}) ...')
    return reviewer_answered.wait(timeout=1.5)   # True only if they answered in time

executed.clear()
model = ScriptedModel([[{'name': 'run_tests', 'args': {}}],
                       [{'name': 'commit', 'args': {'message': 'fix login bug'}}],
                       'Could not commit: approval was not given.'])
answer, trace = agent_loop(model, repo, 'Ship the login bug fix.', approve=ask_reviewer)
show(trace)

**What happened.** To wait longer, the process must stay alive and hold the
conversation in memory for hours, across a lunch break, a deploy or a restart.
If it dies while waiting, you're back in 3.3. Real approvals (a code review, a
manager's sign-off, a customer's reply) take hours or days.

**The idea it points to:** make waiting for a human a **pause, not a block**.
Save a checkpoint, let the process exit, and resume from that checkpoint
whenever the answer arrives, even in a different process on a different machine.

### 3.5 You can't inspect, test, or replay one step

Suppose the report in 3.3 came out wrong and you suspect the third fetch. You'd
like to (a) look at the exact state just before it, (b) change one thing, and
(c) re-run **from that point only**. Does the loop let you?

In [ ]:
print('agent_loop', inspect.signature(agent_loop))
print('\nWhat the report run in 3.3 left behind: a log of text, not resumable state:')
show(report_trace)

**What happened.** `agent_loop` takes a task and nothing else, so every run
starts from the beginning. The trace tells you *what happened*, but not the
state *at each point*, and there's no way to start from step 3. So:

- **Debugging** means re-running everything, with a non-deterministic model that
  may not even repeat the bug.
- **Testing** is all-or-nothing: there's no "step 3" to unit-test, because steps
  aren't named units. They're just iterations of a loop.
- **Auditing** ("why did it merge without approval?") means reading prose.

**The idea it points to:** named **nodes** you can test on their own, plus a
checkpoint at every step, which gives you **time travel**: load the state at
step 3, edit it, and fork a new run from there.

## 4 · The fix in one picture: state + nodes + edges + checkpoints

Every failure above came from the same two things: the model decides the control
flow, and the work lives in RAM. Orchestration moves both into the open:

```
                 ┌────────────────────── State ───────────────────────┐
                 │ { task, notes: [...], tests_passed, approved, ... } │
                 └──────────────▲───────────────────────────┬─────────┘
                    each node   │ reads & returns updates   │ saved after every step
                                │                           ▼
   START ──► fetch_a ─┐                                  ┌──────────────┐
          ├► fetch_b ─┼──► write_report ──► run_tests ──►│ Checkpointer │
          └► fetch_c ─┘         ▲              │         │ (SQLite, …)  │
             (fan-out)          │   failed     │ passed  └──────────────┘
                                └──────────────┤
                                               ▼
                                     human_approval (pause) ──► commit ──► END
```

| Building block | What it is | Fixes |
|---|---|---|
| **State** | One explicit, serialisable object every step reads and updates | 3.3, 3.5: there's something to save and inspect |
| **Node** | A named step: plain code *or* an LLM / agent call | 3.5: testable, replayable units |
| **Edge** | "After this node, go there": fixed, conditional (a router), or a fan-out | 3.1, 3.2: order and parallelism become structure |
| **Checkpointer** | Saves state + position after each step, keyed by a run id (a *thread*) | 3.3, 3.4, 3.5: resume, pause, time travel |

The model still makes the *judgement calls*, inside nodes and in conditional
edges where you want it to choose. Orchestration only takes away the decisions
that should never have been left to chance.

Here's the smallest possible taste: an ordered, checkpointed runner in fifteen
lines. It's deliberately naive; notebook 02 builds a real engine.

In [ ]:
CKPT = pathlib.Path('checkpoints/01_toy.json')
CKPT.parent.mkdir(exist_ok=True)
CKPT.unlink(missing_ok=True)                   # start fresh each time this cell runs

def run_workflow(steps, state, ckpt):
    '''Run steps in a FIXED order, saving state + progress after each one.'''
    saved = json.loads(ckpt.read_text()) if ckpt.exists() else {'done': [], 'state': state}
    for name, fn in steps:
        if name in saved['done']:
            print(f'  skip {name:<13} (checkpointed)')
            continue
        print(f'  run  {name}')
        saved['state'] = fn(saved['state'])
        saved['done'].append(name)
        ckpt.write_text(json.dumps(saved))     # ← durable progress
    return saved['state']

fetched.clear()
crash_next_write = True

def fetch_step(topic):
    def step(state):
        fetch(topic)                            # the same tool as in 3.3
        return {**state, 'notes': state['notes'] + [f'notes on {topic}']}
    return step

def report_step(state):
    return {**state, 'report': write_report()}  # crashes the first time

STEPS = [('fetch_ckpt',   fetch_step('checkpointing')),
         ('fetch_retry',  fetch_step('retries')),
         ('fetch_fan',    fetch_step('fan-out')),
         ('write_report', report_step)]

print('attempt 1:')
try:
    run_workflow(STEPS, {'notes': []}, CKPT)
except ProcessCrash as err:
    print('  💥', err)

print('attempt 2 (a brand-new call, as if the process had restarted):')
final = run_workflow(STEPS, {'notes': []}, CKPT)
print('\nfinal state:', final)
print('fetch() ran for:', fetched, '← each source fetched once, not twice')

Two things to notice:

1. **Order is structural.** `write_report` comes after the fetches because the
   list says so, not because anyone was asked nicely.
2. **The state had to be JSON.** A checkpoint is written to disk and read back,
   possibly by a different process. Anything in state must survive that round
   trip, which is why an open database connection or an SDK client never goes
   into orchestration state. You'll meet this constraint in every framework.

What the toy *can't* do yet: branch on a result, run the fetches in parallel,
let an LLM choose the next step, pause for a human, or keep separate runs apart.
Notebook 02 adds all of that.

## 5 · Trade-offs: don't orchestrate everything

Orchestration isn't free. It trades the loop's flexibility for guarantees.

| A plain agent loop is fine when… | Reach for orchestration when… |
|---|---|
| The task is short (seconds, a handful of steps) | The task runs for minutes, hours, or days |
| You can't predict the steps (open-ended exploration) | Some steps and orderings are known in advance |
| Restarting from scratch is cheap and harmless | Steps are expensive, slow, or have side effects |
| One user, one process, interactive | Humans approve, or other systems call back later |
| You're prototyping | You must test, audit, and debug it in production |

**What you pay:**
- **More design up front.** You decide the state schema and the steps before running anything.
- **Rigidity.** A fixed graph can't do what nobody drew. Good designs keep an agent *inside* a node for the open-ended parts.
- **Serialisation discipline.** Everything in state must be storable.
- **Another layer to learn and debug**, especially with a framework.

The rule of thumb this whole phase returns to: **put determinism where you need
guarantees, and leave the model free where you need judgement.** Notebook 03
turns that into a spectrum of designs.

## 6 · Common mistakes

- **Rules only in the prompt.** "Always run tests first" in a system prompt is a
  preference. If breaking it is unacceptable, make it an edge or a gate in code.
- **Asking the LLM to route what code can route.** If "tests failed → go to the
  fix step" is a simple `if`, write the `if`. It's free, instant, and never wrong.
- **Confusing retries with durability.** Retrying a flaky call is good hygiene,
  but it doesn't survive the process dying. Only a checkpoint does.
- **Putting unserialisable objects in state** (clients, file handles, lambdas).
  It works until the first checkpoint, then fails. Keep *data* in state and
  *resources* outside it.
- **Adopting a framework before naming the problem.** Check which of the five
  gaps you actually have. If the answer is "none", a loop is the better design.

## 7 · How the frameworks in this phase name these ideas

Each section of Phase 5 is a different answer to the same five gaps. Notice how
each one describes the **structure**:

| Section | You describe the structure as… |
|---|---|
| 01 LangGraph | a `StateGraph` of nodes and edges over a typed state |
| 02 OpenAI Agents SDK | agents that **hand off** to each other; the rest is your own Python |
| 03 CrewAI | **crews** (agents + tasks) driven by event-based **Flows** |
| 04 Microsoft Agent Framework | **workflows** of executors connected by edges |
| 05 Google ADK | a tree of agents, with Sequential / Parallel / Loop workflow agents |
| 06 Temporal | a deterministic **workflow** function whose side effects are **activities** |

Once you've built the engine yourself in notebook 02, each framework reads as
"that's their name for a node / an edge / a checkpoint".

## 8 · Try it live (optional, Ollama)

The scripted model forced the failure in 3.1. Does a real model respect the
rule? Below, a real Ollama model drives the *same* `agent_loop`, since any
function that returns a `Turn` can be the model. Run it a few times. It may get
it right, and that's the point: **"usually right" is a probability, and a
production rule needs a guarantee.**

In [ ]:
import os
import ollama

MODEL = os.getenv('OLLAMA_MODEL', 'qwen3-coder:30b')   # any Ollama model that supports tools

def ollama_model(messages, tools):
    msg = ollama.chat(model=MODEL, messages=messages, tools=tools,
                      options={'temperature': 0.7}).message
    return Turn(text=msg.content or '',
                tool_calls=[{'name': c.function.name, 'args': dict(c.function.arguments)}
                            for c in (msg.tool_calls or [])])

try:
    ollama.list()
    executed.clear()
    answer, trace = agent_loop(ollama_model, repo,
                               'The login fix is staged. Ship it; we are in a hurry, just commit.',
                               system=RULE, max_steps=4)
    show(trace)
    ok = 'commit' not in executed or ('run_tests' in executed
                                      and executed.index('run_tests') < executed.index('commit'))
    print('\ntools executed:', executed, '→', 'rule respected' if ok else 'RULE BROKEN')
except Exception as err:
    print('Ollama not reachable, skipping the live run:', err)

## 9 · Check yourself

Try to answer before opening each one.

<details><summary><b>1.</b> Your agent must never call <code>deploy</code> unless <code>run_tests</code> passed in the same run. Where should that rule live, and why is the system prompt not enough?</summary>

In the **structure**: an edge that reaches `deploy` only from a "tests passed" branch (or, at minimum, a code gate before `deploy`). A prompt is input to a probabilistic model, so it makes a violation *unlikely*, not *impossible*. Anything that must *never* happen belongs in code.
</details>

<details><summary><b>2.</b> A 40-step research agent built on this loop crashes at step 38. What is lost, and what would you need to have saved to resume at step 38?</summary>

Everything: all tool results and model turns lived in the local `messages` list. To resume you need, after every step, the **state** (messages / gathered results) *and* the **position** (what runs next), written to durable storage and keyed by a run id. That is exactly what a checkpointer stores.
</details>

<details><summary><b>3.</b> Why did three 1-second lookups take ~3s in the loop, even though they were independent?</summary>

Because concurrency is a property of the **control flow**, not of the tools. The loop is written as `for call in tool_calls: run(call)`, so it runs calls one by one. Parallelism has to be designed into the structure (fan-out / fan-in).
</details>

<details><summary><b>4.</b> Why is "retry the failed call three times" not the same as durability?</summary>

A retry happens *inside a running process* and covers one call. If the process itself dies (deploy, OOM, crash), the retry logic dies with it, and so does all progress. Durability means progress is saved outside the process, so a *new* process can continue.
</details>

<details><summary><b>5.</b> Name two situations where adding orchestration would be the wrong call.</summary>

For example: a chat assistant answering a question with one or two tool calls; an exploratory prototype whose steps you can't predict yet; any short task where restarting is cheap and has no side effects. There the loop's flexibility is worth more than the guarantees.
</details>

<details><summary><b>6.</b> In the toy runner, why would storing an open SQLite connection inside <code>state</code> break the design?</summary>

State is written to disk (here as JSON) and read back, possibly by a different process. A connection is a live resource, not data: it can't be serialised, and even if it could, it would be meaningless in another process. Keep data in state and resources outside it.
</details>

## Takeaway

A plain agent loop hides two decisions: **the model owns the control flow**, and
**the work lives in one process's memory**. That's why it can't guarantee order,
run things in parallel, survive a crash, wait for a human, or replay a step.
Orchestration makes both explicit (**state, nodes, edges, checkpoints**) while
leaving the model in charge of the judgement calls inside them.

Next → `02_building_blocks.ipynb`: build a real orchestration engine (routing,
parallel branches, reducers, checkpoints, pause/resume, time travel) in about
120 lines of plain Python.